In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Reset YelpConnect Trust & Safety
# MAGIC
# MAGIC Limpia productos derivados para ejecutar el pipeline
# MAGIC completamente desde RAW.
# MAGIC
# MAGIC IMPORTANTE:
# MAGIC NO elimina los archivos fuente del Volume RAW.

# COMMAND ----------

CATALOG = "workspace"

BRONZE = f"{CATALOG}.yelp_bronze"
SILVER = f"{CATALOG}.yelp_silver"
GOLD = f"{CATALOG}.yelp_gold"
GOV = f"{CATALOG}.yelp_gov"

print("=" * 72)
print("RESET YELPCONNECT TRUST & SAFETY")
print("=" * 72)

# COMMAND ----------

# ============================================================
# 1. GOLD
# ============================================================

gold_tables = [
    "gld_kpi_trust_safety_rule",
    "gld_kpi_trust_safety_overview",
    "gld_kpi_trust_safety_daily",
    "gld_kpi_trust_safety_business",
    "gld_investigation_queue",
    "gld_review_risk",
    "gld_fact_review_evaluation",
    "gld_dim_evaluation_run",
    "gld_dim_rule",
    "gld_dim_user",
    "gld_dim_business"
]

for table in gold_tables:
    spark.sql(
        f"DROP TABLE IF EXISTS {GOLD}.{table}"
    )
    print(f"✔ DROP {GOLD}.{table}")

# COMMAND ----------

# ============================================================
# 2. SILVER
# ============================================================

silver_tables = [
    # Nuevas T&S
    "slv_review_risk",
    "slv_review_evaluation",
    "slv_review_features",

    # Antigua / obsoleta
    "slv_review_signal_summary",

    # Silver core
    "slv_review_cuarentena",
    "slv_checkin_event",
    "slv_tip",
    "slv_review",
    "slv_user",
    "slv_business"
]

for table in silver_tables:
    spark.sql(
        f"DROP TABLE IF EXISTS {SILVER}.{table}"
    )
    print(f"✔ DROP {SILVER}.{table}")

# COMMAND ----------

# ============================================================
# 3. BRONZE
# ============================================================

bronze_tables = [
    "brz_checkin",
    "brz_tip",
    "brz_review",
    "brz_user",
    "brz_business"
]

for table in bronze_tables:
    spark.sql(
        f"DROP TABLE IF EXISTS {BRONZE}.{table}"
    )
    print(f"✔ DROP {BRONZE}.{table}")

# COMMAND ----------

# ============================================================
# 4. GOBIERNO OPERATIVO
#
# Se reconstruyen mediante:
#
# 01_catalogo_reglas_dq
# 02_catalogo_reglas_ts
# DQ Bronze / Silver
# Trust & Safety
# ============================================================

gov_tables = [
    "evaluation_run",
    "evaluation_rule_version",
    "evaluation_rule",
    "dq_resultados",
    "dq_reglas"
]

for table in gov_tables:
    spark.sql(
        f"DROP TABLE IF EXISTS {GOV}.{table}"
    )
    print(f"✔ DROP {GOV}.{table}")

# COMMAND ----------

# ============================================================
# 5. NO BORRAR ESTOS OBJETOS DE GOBIERNO
# ============================================================

print()
print("SE CONSERVAN:")
print("✔ workspace.yelp_gov.diccionario_datos")
print("✔ workspace.yelp_gov.glosario_negocio")
print("✔ workspace.yelp_gov.matriz_raci")

# COMMAND ----------

# ============================================================
# 6. LIMPIAR CHECKPOINT AUTO LOADER
#
# CRÍTICO PARA REPROCESAR REVIEW DESDE RAW.
# ============================================================

CHECKPOINT_REVIEW = (
    "/Volumes/workspace/yelp_bronze/raw/_checkpoints/review"
)

try:
    dbutils.fs.rm(
        CHECKPOINT_REVIEW,
        True
    )

    print(
        f"✔ Checkpoint eliminado: "
        f"{CHECKPOINT_REVIEW}"
    )

except Exception as exc:

    print(
        "⚠ No se encontró o no pudo eliminarse "
        "el checkpoint Review."
    )

    print(exc)

# COMMAND ----------

# ============================================================
# 7. VALIDAR RAW
# ============================================================

RAW = (
    "/Volumes/workspace/yelp_bronze/raw"
)

print()
print("Contenido RAW conservado:")

display(
    dbutils.fs.ls(
        RAW
    )
)

# COMMAND ----------

print()
print("=" * 72)
print("RESET FINALIZADO")
print("=" * 72)

print()
print("Las tablas derivadas fueron eliminadas.")
print("Los archivos RAW fueron conservados.")
print()
print("Siguiente paso:")
print("00_configurar_lakehouse / catálogo / Bronze")
print("=" * 72)